# Create DRAGON Dataset for DeReC Fine-tuning

DRAGON has no explicit "grounded/ungrounded" labels, so we create a synthetic evaluation dataset:
- **Grounded**: answers via normal RAG pipeline (label=1)
- **Ungrounded**: LLM answers WITHOUT context (label=0)

We iterate over all dataset versions (1.0.0 → 1.15.0) to collect as many unique questions as possible.
Each version's questions are processed with that version's texts (retriever rebuilt per version).
Published to HuggingFace for DeReC fine-tuning.

## 1. Setup

In [1]:
!git clone -b feature/evaluate-fact-checking https://github.com/BigMak1/rag_fact_checking.git
!git -C rag_fact_checking pull

fatal: destination path 'rag_fact_checking' already exists and is not an empty directory.


Already up to date.


In [2]:
%%time
%%capture

!pip install -q -r rag_fact_checking/DRAGON/requirements.txt

CPU times: user 9.89 ms, sys: 8.99 ms, total: 18.9 ms
Wall time: 5.19 s


In [3]:
import json
import random
import os
from collections import defaultdict
import gc

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

from datasets import load_dataset, Dataset, DatasetDict
from langchain_community.llms import VLLM
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np
import torch
from transformers import AutoTokenizer
from tqdm import tqdm

from rag_fact_checking.DRAGON.rag_bench import baseline, data, results
from rag_fact_checking.DRAGON.rag_bench.helper import get_ds_versions, sort_versions
from rag_fact_checking.DRAGON.rag_bench.constants import HIST_TEXTS_REPO_ID, HIST_QUESTIONS_REPO_ID

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [5]:
# ── Constants ──
HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"
RANDOM_SEED: int = 42
EMBEDDER_NAME: str = "ai-forever/FRIDA"
LLM_NAME: str = "bond005/meno-tiny-0.1"

HF_TOKEN: str = user_secrets.get_secret("HF_TOKEN")
HF_DATASET_REPO: str = "Makson4ic/dragon-derec-dataset"

# Version range to process
MIN_VERSION: str = "1.0.0"
MAX_VERSION: str = "1.15.0"

# ── Prompts ──
LLM_PROMPT: str = """Проанализируйте заданный контекст и ответьте на вопрос пользователя на основе сведений, предоставленных в этом контексте.
Не давайте никаких объяснений и пояснений к своему ответу. Не пишите ничего лишнего. Не извиняйтесь, не стройте диалог. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
Если в заданном контексте нет информации для ответа на вопрос пользователя, то ничего не придумывайте и просто откажитесь отвечать.
"""

LLM_PROMPT_NO_CONTEXT: str = """Ответьте на вопрос пользователя.
Не давайте никаких объяснений. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
"""

In [6]:
# ── Helpers ──
def _build_question_index(questions_ds):
    """Map str(question_id) -> {"question": ...}"""
    idx = {}
    for item in questions_ds["train"]:
        idx[str(item["id"])] = {"question": item["question"]}
    return idx


def _build_text_index(texts_ds):
    """Map doc_id -> text content"""
    idx = {}
    for item in texts_ds["train"]:
        idx[item["id"]] = item["text"]
    return idx


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version)


def version_tuple(v):
    """Convert version string to tuple for comparison."""
    return tuple(int(x) for x in v.split("."))

In [7]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.random.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

## 2. Load Models & Discover Versions

In [8]:
text_versions = set(get_ds_versions(HIST_TEXTS_REPO_ID))
question_versions = set(get_ds_versions(HIST_QUESTIONS_REPO_ID))
common_versions = text_versions & question_versions

# Filter to [MIN_VERSION, MAX_VERSION] range
min_t, max_t = version_tuple(MIN_VERSION), version_tuple(MAX_VERSION)
versions = sort_versions([
    v for v in common_versions
    if min_t <= version_tuple(v) <= max_t
])

print(f"Found {len(versions)} versions: {versions}")

Found 15 versions: ['1.0.0', '1.1.0', '1.2.0', '1.3.0', '1.4.0', '1.5.0', '1.6.0', '1.7.0', '1.8.0', '1.9.0', '1.10.0', '1.11.0', '1.12.0', '1.13.0', '1.15.0']


In [9]:
# Kaggle FIX
os.environ["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
print("LIBRARY_PATH =", os.environ["LIBRARY_PATH"])

llm = VLLM(
    model=LLM_NAME,
    tensor_parallel_size=1,
    max_new_tokens=256,
    top_p=0.95,
    temperature=0.3,
    vllm_kwargs={
        "gpu_memory_utilization": 0.45,
        "max_num_batched_tokens": 8192,
        "max_model_len": 4096,
        "disable_log_stats": True,
        "seed": RANDOM_SEED
    },
    disable_log_stats=True,
)
tok = AutoTokenizer.from_pretrained(LLM_NAME)

LIBRARY_PATH = /usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs


2026-03-12 15:29:02.424900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773329342.450101     451 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773329342.457587     451 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773329342.475994     451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773329342.476016     451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773329342.476019     451 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-03-12 15:29:23.618280: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773329363.646077     506 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773329363.653608     506 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773329363.671665     506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773329363.671690     506 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773329363.671693     506 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=506) ERROR 03-12 15:29:34 [fa_utils.py:131] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100] EngineCore failed to start.
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100] Traceback (most recent call last):
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100]   File "/usr/local/lib/python3.12/dist-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/compiler.py", line 148, in compile
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100]     pm.run(module.operation)
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100] cutlass._mlir._mlir_libs._site_initialize.<locals>.MLIRError: Failure while executing pass pipeline:
(EngineCore_DP0 pid=506) ERROR 03-12 15:29:35 [core.py:1100] error: unknown: failed to verify the compilation unit (error 7: NVVM_ERROR_INVALID_OPTION), libNVVM extra log: libnvvm : error: -arc

(EngineCore_DP0 pid=506) Process EngineCore_DP0:
(EngineCore_DP0 pid=506) Traceback (most recent call last):
(EngineCore_DP0 pid=506)   File "/usr/local/lib/python3.12/dist-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/compiler.py", line 148, in compile
(EngineCore_DP0 pid=506)     pm.run(module.operation)
(EngineCore_DP0 pid=506) cutlass._mlir._mlir_libs._site_initialize.<locals>.MLIRError: Failure while executing pass pipeline:
(EngineCore_DP0 pid=506) error: unknown: failed to verify the compilation unit (error 7: NVVM_ERROR_INVALID_OPTION), libNVVM extra log: libnvvm : error: -arch=compute_ is an unsupported option  
(EngineCore_DP0 pid=506)  note: unknown: see current operation: 
(EngineCore_DP0 pid=506)   "gpu.module"() <{sym_name = "kernels", targets = [#nvvm.target<O = 3, chip = "", flags = {"ptx-cmd-options" = []}>]}> ({
(EngineCore_DP0 pid=506)   ^bb0:
(EngineCore_DP0 pid=506)   }) {compute_targets = [#cuda.compute_target<sass, portable, []>]} : () -> ()
(Engin

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDER_NAME,
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"batch_size": 16, "prompt": "search_document: "},
    query_encode_kwargs={"prompt": "search_query: "}
)

# Build no-context prompt template (same for all versions)
messages_no_ctx = [
    {"role": "system", "content": LLM_PROMPT_NO_CONTEXT},
    {"role": "user", "content": "Вопрос: {question}"},
]
template_no_ctx = tok.apply_chat_template(
    messages_no_ctx, tokenize=False, add_generation_prompt=True
)

## 3. Generate Answers for All Versions

For each version (newest first):
1. Load texts and questions for that version
2. Skip questions already processed in a newer version
3. Build retriever on that version's texts
4. Generate **grounded** (with context) and **ungrounded** (without context) answers
5. Collect dataset records directly

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDER_NAME,
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"batch_size": 16, "prompt": "search_document: "},
    query_encode_kwargs={"prompt": "search_query: "}
)

# Build no-context prompt template (same for all versions)
messages_no_ctx = [
    {"role": "system", "content": LLM_PROMPT_NO_CONTEXT},
    {"role": "user", "content": "Вопрос: {question}"},
]
template_no_ctx = tok.apply_chat_template(
    messages_no_ctx, tokenize=False, add_generation_prompt=True
)

## 4. Save Full Dataset

In [ ]:
results.save(dataset_records, "./fact_check_eval_dataset.json")
print(f"Saved fact-checking dataset: {len(dataset_records)} examples")
print(f"  Grounded: {sum(1 for r in dataset_records if r['is_grounded'])}")
print(f"  Ungrounded: {sum(1 for r in dataset_records if not r['is_grounded'])}")

## 5. Train/Val/Test Split

Split by `question_id` so that both grounded and ungrounded examples for the same question stay in the same split. Proportions: 70% train / 15% val / 15% test.

In [ ]:
# Train/val/test split by question_id
# Same question's grounded + ungrounded examples stay in the same split
unique_qids = list(set(r["question_id"] for r in dataset_records))
random.shuffle(unique_qids)

n_total = len(unique_qids)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_qids = set(unique_qids[:n_train])
val_qids = set(unique_qids[n_train:n_train + n_val])
test_qids = set(unique_qids[n_train + n_val:])

train_records = [r for r in dataset_records if r["question_id"] in train_qids]
val_records = [r for r in dataset_records if r["question_id"] in val_qids]
test_records = [r for r in dataset_records if r["question_id"] in test_qids]

print(f"Train: {len(train_records)} examples ({len(train_qids)} questions)")
print(f"Val:   {len(val_records)} examples ({len(val_qids)} questions)")
print(f"Test:  {len(test_records)} examples ({len(test_qids)} questions)")
print(f"Total: {len(train_records) + len(val_records) + len(test_records)}")

In [ ]:
# Save splits locally
import os as _os

save_dir = "rag_fact_checking/DEREC/dataset/DRAGON"
_os.makedirs(save_dir, exist_ok=True)

results.save(train_records, f"{save_dir}/train.json")
results.save(val_records, f"{save_dir}/val.json")
results.save(test_records, f"{save_dir}/test.json")

print(f"Saved to {save_dir}/")
print(f"  train.json: {len(train_records)} examples")
print(f"  val.json:   {len(val_records)} examples")
print(f"  test.json:  {len(test_records)} examples")

## 6. Publish Dataset to HuggingFace

Upload train/val/test splits to HuggingFace Hub so that `train_dragon.ipynb` can load them via `load_dataset()`.

In [ ]:
ds_dict = DatasetDict({
    "train": Dataset.from_list(train_records),
    "val": Dataset.from_list(val_records),
    "test": Dataset.from_list(test_records),
})

ds_dict.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)
print(f"Published to https://huggingface.co/datasets/{HF_DATASET_REPO}")
print(f"  train: {len(train_records)}, val: {len(val_records)}, test: {len(test_records)}")